# SEARLE: bộ chạy Colab của nhóm

Mỗi phiên Colab: chọn GPU (*Thời gian chạy → Thay đổi loại thời gian chạy*), rồi chạy lần lượt **ô 0 → ô 5**.
Sau đó dán lệnh của nhiệm vụ vào **mẫu ô lệnh** ở cuối notebook.

- Lần đầu: *Tệp → Lưu bản sao vào Drive*, rồi sửa `RUNNER` ở ô 0 thành tên của mình.
- Kế hoạch và lệnh từng nhiệm vụ: `plans/nhiem_vu_Chi.md` (đánh giá, thí nghiệm) và `plans/nhiem_vu_train_Colab.md` (train) trong repo.
- Kết quả mỗi lượt chạy có `--output-dir` nằm ở `$DRIVE/results/<mã>/`; nhật ký tự động ở `$DRIVE/results/runs_<RUNNER>.csv`.

### Ô 0. Cấu hình

In [ ]:
RUNNER = "Chi"        # đổi thành "Hoa" hoặc "Doanh"
DATASETS = "cirr"     # các tập cần chuẩn bị ở ô 4, cách nhau bằng dấu cách: cirr circo fiq imagenet
BRANCH = "main"

### Ô 1. Gắn Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
assert os.path.isdir('/content/drive/MyDrive/SEARLE'), \
    "Chưa thấy MyDrive/SEARLE: mở link thư mục SEARLE → chuột phải → Sắp xếp → Thêm lối tắt → Drive của tôi"
print("Drive OK")

### Ô 2. Lấy code mới nhất từ GitHub

In [ ]:
import os
REPO = "/content/XLA_SEARLE"
if os.path.isdir(f"{REPO}/.git"):
    !git -C {REPO} remote set-url origin https://github.com/chilk1307/Chiss.git
    !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} reset -q --hard origin/{BRANCH}
else:
    !git clone -q -b {BRANCH} https://github.com/chilk1307/Chiss.git {REPO}
!git -C {REPO} log -1 --format="Code: %h  %s  (%cr)"

### Ô 3. Cài môi trường (Python 3.8, vài phút ở lần đầu mỗi phiên)

In [ ]:
!bash /content/XLA_SEARLE/SEARLE-main/colab/setup_env.sh {RUNNER}

### Ô 4. Chuẩn bị dữ liệu (giải nén từ Drive ra `/content/data`)

In [ ]:
!bash /content/XLA_SEARLE/SEARLE-main/colab/prepare_data.sh {DATASETS}

### Ô 5. Kiểm tra GPU

In [ ]:
%%bash
source /content/env.sh
nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Không có GPU: Thời gian chạy → Thay đổi loại thời gian chạy"
$PY -c "import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())"

---
## Kiểm thử bộ chạy (C2, chạy một lần khi có CIRR)

Kỳ vọng: `cirr_recall_at1 = 25.66` (±0.05). Lần đầu tải trọng số SEARLE qua `torch.hub` (cần internet).

In [ ]:
%%bash
source /content/env.sh
cd $SRC
time $PY src/validate.py --eval-type searle --dataset cirr --dataset-path $DATA/CIRR

## Kiểm tra dữ liệu (E1.2)

Chạy cho các tập đã chuẩn bị ở ô 4; kết quả ở `$DRIVE/results/E1.2-dataset-check/report.md`.

In [ ]:
%%bash
source /content/env.sh
cd $SRC
for DS in cirr circo fashioniq; do
  P=$DATA/CIRR; [ $DS = circo ] && P=$DATA/CIRCO; [ $DS = fashioniq ] && P=$DATA/FashionIQ
  if [ -d $P ]; then $PY tools/check_datasets.py --dataset $DS --dataset-path $P --out $DRIVE/results/E1.2-dataset-check; fi
done

---
## Mẫu ô lệnh

Chép ô này cho mỗi nhiệm vụ, dán lệnh vào cuối. Mỗi lượt đánh giá nên có `--cache-dir $DRIVE/cache` và
`--output-dir $DRIVE/results/<mã thí nghiệm>`; thêm `--note "..."` nếu cần ghi chú.

In [ ]:
%%bash
source /content/env.sh        # khai báo DRIVE, DATA, PY, RUNNER
cd /content/XLA_SEARLE/SEARLE-main
# Ví dụ:
# $PY src/validate.py --eval-type searle --dataset cirr --dataset-path $DATA/CIRR --cache-dir $DRIVE/cache --output-dir $DRIVE/results/E1.7a-searle-cirr

## Chạy nền (train dài)

`nohup ... &` để lệnh chạy tiếp khi ô kết thúc; theo dõi bằng ô thứ hai. Colab ngắt thì chạy lại ô 0–5 rồi chạy đúng lệnh cũ, thêm `--resume`.

In [ ]:
%%bash
source /content/env.sh
cd /content/XLA_SEARLE/SEARLE-main
# mkdir -p $DRIVE/results/<mã>
# nohup $PY src/train_phi.py ... > $DRIVE/results/<mã>/train.log 2>&1 &

In [ ]:
%%bash
source /content/env.sh
# tail -n 3 $DRIVE/results/<mã>/train.log
nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

## Xem kết quả đã có

In [ ]:
%%bash
source /content/env.sh
cd /content/XLA_SEARLE/SEARLE-main
$PY tools/collect_results.py --results-dir $DRIVE/results --out $DRIVE/results/summary.md > /dev/null
tail -n 40 $DRIVE/results/summary.md
echo; echo "--- 5 dòng cuối runs_$RUNNER.csv ---"
tail -n 5 $DRIVE/results/runs_$RUNNER.csv 2>/dev/null || echo "(chưa có lượt chạy nào)"